# Deep Agents + Ollama (gemma4:e4b) + PostgreSQL Persistent Memory

Runs a `deepagents` agent with a **local Ollama model** and stores chat memory in a  
**local PostgreSQL database** (via pgAdmin 4), scoped by **user_id** and **thread_id**.

### Prerequisites
```bash
pip install deepagents "langgraph[postgres]" python-dotenv langchain-ollama psycopg psycopg-binary httpx
ollama pull gemma4:e4b
```

### `.env` file
Create a `.env` file next to this notebook:
```
PG_DATABASE_URL=postgresql://postgres:<your_password>@localhost:5432/<your_db_name>
```
> **pgAdmin 4 tip:** Find your port under pgAdmin → Servers → right-click → Properties → Connection tab.  
> Format: `postgresql://<username>:<password>@localhost:<port>/<database>`

> **Ollama tip:** Make sure Ollama is running (`ollama serve`) before executing the cells below.

In [1]:
# ── 1. Install dependencies (run once) ───────────────────────────────────────
%pip install -q deepagents "langgraph[postgres]" python-dotenv langchain-ollama psycopg psycopg-binary httpx

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import os
import httpx
from dotenv import load_dotenv

from langchain_ollama import ChatOllama
from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.postgres import PostgresStore

load_dotenv()

PG_URL = os.environ["PG_DATABASE_URL"]

print("Environment loaded")
print(f"  DB: {PG_URL.split('@')[-1]}")  # print only host/db, not password

Environment loaded
  DB: localhost:5432/Chat_memory


In [3]:
# ── 3. Verify Ollama is running and the model is available ───────────────────
OLLAMA_MODEL = "gemma4:e4b"

try:
    r = httpx.get("http://localhost:11434/api/tags", timeout=5)
    available = [m["name"] for m in r.json().get("models", [])]
    print("Ollama running. Available models:", available)
    if OLLAMA_MODEL not in available:
        print(f"\n  Model '{OLLAMA_MODEL}' not pulled yet.")
        print(f"  Run in a terminal:  ollama pull {OLLAMA_MODEL}")
    else:
        print(f"  '{OLLAMA_MODEL}' is ready.")
except Exception as e:
    print("Ollama not reachable:", e)
    print("  Start it with:  ollama serve")
    raise

Ollama running. Available models: ['gemma4:e4b', 'nomic-embed-text:latest', 'llama3.1:8b']
  'gemma4:e4b' is ready.


In [4]:
# ── 4. Verify PostgreSQL connectivity ────────────────────────────────────────
import psycopg

try:
    with psycopg.connect(PG_URL) as conn:
        row = conn.execute("SELECT version();").fetchone()
        print("Connected to PostgreSQL:", row[0][:70])
except Exception as e:
    print("Connection failed:", e)
    print("\nCheck your PG_DATABASE_URL in .env")
    print("Format: postgresql://postgres:<password>@localhost:5432/<dbname>")
    raise

Connected to PostgreSQL: PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35226, 64-bi


In [5]:
# ── 5. Namespace factory (user_id + thread_id scoping) ───────────────────────
#
# The namespace tuple becomes the Postgres storage key:
#   ("<user_id>", "<thread_id>", "memories")
#
# Resolution order for user_id:
#   1. Server-injected identity  → production (LangSmith / LangGraph Cloud)
#   2. context.user_id           → local override passed via config
#   3. "local_user"              → anonymous fallback

def make_namespace(rt):
    """
    Receives the deepagents Runtime and returns a 3-tuple namespace.
    Two users with the same thread_id still get isolated memory rows.
    """
    # --- user_id ---
    if rt.server_info and rt.server_info.user and rt.server_info.user.identity:
        user_id = rt.server_info.user.identity           # production
    else:
        user_id = getattr(rt.context, "user_id", "local_user")  # local

    # --- thread_id (always available) ---
    thread_id = rt.execution_info.thread_id

    return (user_id, thread_id, "memories")

print("Namespace factory defined")

Namespace factory defined


In [6]:
# ── 6. Build model + store + agent ───────────────────────────────────────────

# --- 6a. Ollama local model --------------------------------------------------
model = ChatOllama(
    model="gemma4:e4b",
    temperature=0.3,
)
print("ChatOllama model ready (gemma4:e4b)")

# --- 6b. PostgresStore (open once, keep alive for the session) ---------------
_store_ctx = PostgresStore.from_conn_string(PG_URL)
postgres_store = _store_ctx.__enter__()
postgres_store.setup()   # creates langgraph store tables if they don't exist
print("PostgresStore initialised")

# --- 6c. Backend: /memories/ -> Postgres, everything else -> ephemeral state -
def make_backend(rt):
    return CompositeBackend(
        default=StateBackend(rt),           # ephemeral working files
        routes={
            "/memories/": StoreBackend(     # persistent, scoped by user+thread
                rt,
                store=postgres_store,
                namespace=make_namespace,   # callable, NOT the result of calling it
            )
        },
    )

# --- 6d. Create the agent ----------------------------------------------------
agent = create_deep_agent(
    model=model,
    backend=make_backend,
    store=postgres_store,
    system_prompt=(
        "You are a helpful assistant with persistent memory. "
        "Whenever the user shares personal information, preferences, or important facts, "
        "proactively save them to /memories/profile.txt so you can recall them later. "
        "Before answering questions about the user, read /memories/profile.txt first."
    ),
)
print("Agent created and ready")

ChatOllama model ready (gemma4:e4b)
PostgresStore initialised
Agent created and ready


In [7]:
# ── 7. Helper: send a message as a specific user / thread ────────────────────
from langchain_core.messages import HumanMessage

def chat(
    message: str,
    user_id: str = "alice",
    thread_id: str = "thread-001",
) -> str:
    """
    Invoke the agent with a specific user_id + thread_id.
    Memory stored in /memories/ persists across calls with the same user_id.
    """
    config = {
        "configurable": {
            "thread_id": thread_id,
            # Inject user_id into context so make_namespace() can read it locally
            "context": type("Ctx", (), {"user_id": user_id})(),
        }
    }
    result = agent.invoke(
        {"messages": [HumanMessage(content=message)]},
        config=config,
    )
    reply = result["messages"][-1].content
    print(f"[{user_id} / {thread_id}]  {reply}\n")
    return reply

print("chat() helper ready")

chat() helper ready


In [8]:
# ── 8. Demo: two users, two threads ──────────────────────────────────────────

# Alice introduces herself on thread t1
chat("Hi! My name is Alice and I love Python.", user_id="alice", thread_id="t1")

# The agent should remember within the same thread
chat("What's my name?",                         user_id="alice", thread_id="t1")

c:\Users\gaura\OneDrive\Desktop\AI projects\agentic_ai_advanced\DeepAgents\deepagents\.venv\Lib\site-packages\deepagents\middleware\filesystem.py:1679: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
C:\Users\gaura\AppData\Local\Temp\ipykernel_28984\3396731045.py:19: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and writes state via `get_config()`. Use `StateBackend()` instead.
  default=StateBackend(rt),           # ephemeral working files
C:\Users\gaura\AppData\Local\Temp\ipykernel_28984\3396731045.py:21: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains 

[alice / t1]  /memories/profile.txt has been updated with your name and interest in Python. How can I help you today?



c:\Users\gaura\OneDrive\Desktop\AI projects\agentic_ai_advanced\DeepAgents\deepagents\.venv\Lib\site-packages\deepagents\middleware\filesystem.py:1679: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
C:\Users\gaura\AppData\Local\Temp\ipykernel_28984\3396731045.py:19: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and writes state via `get_config()`. Use `StateBackend()` instead.
  default=StateBackend(rt),           # ephemeral working files
C:\Users\gaura\AppData\Local\Temp\ipykernel_28984\3396731045.py:21: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains 

[alice / t1]  /memories/profile.txt
No profile information found.



'/memories/profile.txt\nNo profile information found.'

In [ ]:
# Bob on the same thread —  (different user_id)
chat("Do you know who I am?", user_id="bob", thread_id="t1")

c:\Users\gaura\OneDrive\Desktop\AI projects\agentic_ai_advanced\DeepAgents\deepagents\.venv\Lib\site-packages\deepagents\middleware\filesystem.py:1679: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
C:\Users\gaura\AppData\Local\Temp\ipykernel_28984\3396731045.py:19: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and writes state via `get_config()`. Use `StateBackend()` instead.
  default=StateBackend(rt),           # ephemeral working files
C:\Users\gaura\AppData\Local\Temp\ipykernel_28984\3396731045.py:21: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains 

[bob / t1]  Yes, I know that your name is Alice and that your interests include Python.



'Yes, I know that your name is Alice and that your interests include Python.'

In [14]:
# Alice on a new thread — StoreBackend persists across threads for the same user
chat("What do you know about me?", user_id="alice", thread_id="t1")

c:\Users\gaura\OneDrive\Desktop\AI projects\agentic_ai_advanced\DeepAgents\deepagents\.venv\Lib\site-packages\deepagents\middleware\filesystem.py:1679: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
C:\Users\gaura\AppData\Local\Temp\ipykernel_28984\3396731045.py:19: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and writes state via `get_config()`. Use `StateBackend()` instead.
  default=StateBackend(rt),           # ephemeral working files
C:\Users\gaura\AppData\Local\Temp\ipykernel_28984\3396731045.py:21: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains 

[alice / t1]  Based on the information stored in my memory, I know that your name is Alice and that your interests include Python.



'Based on the information stored in my memory, I know that your name is Alice and that your interests include Python.'

In [11]:
# ── 9. Inspect stored rows directly in Postgres ──────────────────────────────
with psycopg.connect(PG_URL) as conn:
    rows = conn.execute("""
        SELECT prefix, key, left(value::text, 120) AS value_preview
        FROM store
        ORDER BY prefix, key
        LIMIT 20;
    """).fetchall()

if rows:
    print(f"{'PREFIX':<45} {'KEY':<30} VALUE PREVIEW")
    print("-" * 110)
    for prefix, key, val in rows:
        print(f"{str(prefix):<45} {key:<30} {val}")
else:
    print("No rows yet — the agent may not have written to /memories/ in this session.")

PREFIX                                        KEY                            VALUE PREVIEW
--------------------------------------------------------------------------------------------------------------
local_user.t1.memories                        /profile.txt                   {"content": "Name: Alice\nInterests: Python", "encoding": "utf-8", "created_at": "2026-06-14T16:36:24.977202+00:00", "mo


In [12]:
# Alice on a new thread — StoreBackend persists across threads for the same user
chat("What do you know about me?", user_id="alice", thread_id="t2")

c:\Users\gaura\OneDrive\Desktop\AI projects\agentic_ai_advanced\DeepAgents\deepagents\.venv\Lib\site-packages\deepagents\middleware\filesystem.py:1679: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
C:\Users\gaura\AppData\Local\Temp\ipykernel_28984\3396731045.py:19: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and writes state via `get_config()`. Use `StateBackend()` instead.
  default=StateBackend(rt),           # ephemeral working files
C:\Users\gaura\AppData\Local\Temp\ipykernel_28984\3396731045.py:21: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains 

[alice / t2]  I do not have any personal information stored about you right now.

If you share any personal details, preferences, or important facts, I will proactively save them to my memory so I can recall them for you later.



'I do not have any personal information stored about you right now.\n\nIf you share any personal details, preferences, or important facts, I will proactively save them to my memory so I can recall them for you later.'

In [13]:
# ── 10. Cleanup — close the store when done ──────────────────────────────────
# Run this when you're finished to release the Postgres connection pool.

# _store_ctx.__exit__(None, None, None)
# print("PostgresStore connection closed")

## Troubleshooting

| Error | Fix |
|---|---|
| `connection refused` port 11434 | Ollama isn't running — run `ollama serve` in a terminal |
| `model 'gemma4:e4b' not found` | Run `ollama pull gemma4:e4b` |
| `connection refused` port 5432 | Check pgAdmin 4 is running and the port matches |
| `password authentication failed` | Double-check user/password in `.env` |
| `database does not exist` | Create the DB in pgAdmin first (right-click Databases → Create) |
| `relation "store" does not exist` | Call `postgres_store.setup()` — it creates the table |
| `ModuleNotFoundError: psycopg` | Run `pip install psycopg psycopg-binary` |
| `ModuleNotFoundError: langchain_ollama` | Run `pip install langchain-ollama` |
| `StoreBackend` namespace error | Make sure `namespace=` receives a **callable**, not the result of calling it |

## How the namespace maps to Postgres rows

Every file the agent writes under `/memories/` is stored as:

```
prefix  →  ("<user_id>", "<thread_id>", "memories")
key     →  <filename inside /memories/>
value   →  file content (list of lines)
```

Two users with the **same `thread_id`** still have fully isolated rows because `user_id` is part of the prefix.

## Switching models

To use a different Ollama model, change `model=` in cell 6 and pull it first:
```bash
ollama pull llama3.2
```
Then update the cell:
```python
model = ChatOllama(model="llama3.2", temperature=0.3)
```